# Two-Arm Bandit Task: Raw Data Exploration

**Author:** Kerim Atak

**Date:** January 2026

## Raw Data

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import numpy as np
import json
import numpy as np
import json
import os
from ncmcm.visualisers.twoArmBandit_session_visualization import generate_interactive_session_plot 

print("Current OS Path:", os.getcwd())

### Session ID
Set the session identifier for data loading.

In [ ]:
data_session = "JPAS_0023_20230922"

### Cluster Quality Filtering
Load cluster metadata and filter for high-quality units only.

**Observation:** 367 good-quality neurons with quality metrics (amplitude, contamination %, SNR, silhouette score, etc.).

In [ ]:
# import cluster_info.tsv
## contains information about each cluster (neuron), including quality metrics
## NOTE: This is what will be used to filter good units for NC-MCM/BunDLe-Net analysis
df = pd.read_csv(os.path.join(data_session, "cluster_info.tsv"), sep="\t")

# filter column "group" for value "good" to keep only good units
df = df[df["group"] == "good"]

print(df.head())
print("DataFrame Shape:", df.shape)

### Channel Mapping
Physical channel mapping on the recording probe.

**Observation:** 384 recording channels sequentially indexed.

In [ ]:
# import channel_map.npy
## contains the mapping of channels to their physical locations on the probe ... probably
channel_map = np.load(os.path.join(data_session, "channel_map.npy"), allow_pickle=True)
print("Channel Map Shape:", channel_map.shape)
print("Channel Map Data:", channel_map)

### Channel Positions
Spatial coordinates of each channel on the probe.

**Observation:** 384 channel positions in 2D (x, y coordinates in μm).

In [ ]:
# import channel_positions.npy
## contains the physical positions of each channel on the probe ... probably
channel_positions = np.load(os.path.join(data_session, "channel_positions.npy"), allow_pickle=True)
print("Channel Positions Shape:", channel_positions.shape)
print("Channel Positions Data:", channel_positions)

### Spike Times (Raw)
Timestamps of all spike events in sample units.

**Observation:** 8.7M spikes spanning ~1400 seconds (23 min) at 32 kHz sampling rate.

In [ ]:
# import spike_times.npy
## contains the timestamps of each spike event of all neurons
## NOTE: This is what will be used for NC-MCM/BunDLe-Net analysis
## NOTE: spike_times, spike_clusters, and spike_templates are aligned by index
spike_times = np.load(os.path.join(data_session, "spike_times.npy"))
print("Spike times shape:", spike_times.shape)
print("First 10 spike times:", spike_times[:10])
print("First spike time:", np.min(spike_times))
print("Last spike time:", np.max(spike_times))

# plot histogramm of spike times
import plotly.express as px
fig = px.histogram(spike_times, nbins=100, title="Spike Times Histogram", labels={"value": "Time (samples)", "count": "Number of Spikes"})
fig.show()

In [ ]:
# import spike_clusters.npy
## assigns each spike to a cluster (neuron)
## NOTE: This is what will be used for NC-MCM/BunDLe-Net analysis
spike_clusters = np.load(os.path.join(data_session, "spike_clusters.npy"))
print("Spike clusters shape:", spike_clusters.shape)
print("First 10 spike clusters:", spike_clusters[:10])

In [ ]:
# import spike_templates.npy
## output file from Kilosort, assigns each spike to a template rather than a cluster (neuron), needs further processing/merging to get neuron assignments
spike_templates = np.load(os.path.join(data_session, "spike_templates.npy"))
print("Spike templates shape:", spike_templates.shape)
print("First 10 spike templates:", spike_templates[:10])

In [ ]:
# import spike_times_milliseconds_sync_to_behav_JPAS_0023_20230922.npy
## contains spike times in milliseconds, synchronized to behavioral data (the time values are the same as behavioral time values in metrics.json)
spike_times_ms_sync_to_behav = np.load(os.path.join(data_session, "spike_times_milliseconds_sync_to_behav.npy"))
print("Spike times ms shape:", spike_times_ms_sync_to_behav.shape)
print("First 10 spike times ms:", spike_times_ms_sync_to_behav[:10])

In [ ]:
# import params.py, read as txt and print content
## contains metadata about the recording session
## parse n_channels_dat, offset, sample_rate, dtype, hp_filtered from params.py
## NOTE: Only sample_rate is interessting for now, which is 32050.755862876373 Hz
with open(os.path.join(data_session, "params.py"), "r") as f:
    params_content = f.read()
    print("Params.py content:\n", params_content)
    n_channels_dat = params_content.split("n_channels_dat = ")[1].split("\n")[0]
    offset = params_content.split("offset = ")[1].split("\n")[0]
    sample_rate = params_content.split("sample_rate = ")[1].split("\n")[0]
    dtype = params_content.split("dtype = '")[1].split("'")[0]
    hp_filtered = params_content.split("hp_filtered = ")[1].split("\n")[0]

print(f"n_channels_dat: {n_channels_dat}, offset: {offset}, sample_rate: {sample_rate}, dtype: {dtype}, hp_filtered: {hp_filtered}")

In [ ]:
# import metrics.json as dict and print keys
## contains various metrics about the recording session like 
with open(os.path.join(data_session, "metrics.json"), "r") as f:
    metrics = json.load(f)
    print("Metrics keys:", metrics.keys())

In [ ]:
def _metrics_json_overview(metrics_dict, showcase_num=5):
    # Print an overview of the metrics.json content
    for key, value in metrics_dict.items():
        print(f"{key}: {type(value)} - {value if isinstance(value, (int, float, str)) else '...'}")
        # if key is "metrics", print the keys inside metrics
        if key == "metrics":
            for metric_key in value.keys():
                print(f"  - {metric_key}")
    
    print()
    
    # experiment data
    print("="*15 + " Experiment Data " + "="*15)
    print(json.dumps(metrics_dict.get("experiment data", {}), indent=4))
    print() 
    
    # performance
    print("="*15 + " Performance " + "="*15)
    print(json.dumps(metrics_dict.get("performance", {}), indent=4))
    print()
    
    # metrics - blocks
    print("="*15 + " Metrics - Blocks " + "="*15)
    # num of blocks
    print("Number of blocks:", len(metrics_dict.get("metrics", {}).get("blocks", {})))
    print(json.dumps(metrics_dict.get("metrics", {}).get("blocks", {}), indent=4))
    print() 
    
    # metrics - trials
    print("="*15 + " Metrics - Trials " + "="*15)
    # num of trials
    trials = metrics_dict.get("metrics", {}).get("trials", [])
    print("Number of trials:", len(trials))
    print(f"First {showcase_num} trials:")
    print(json.dumps(trials[:showcase_num], indent=4))  # Print only the first 5 trials
    print("...")
    
    # metrics - states
    print("="*15 + " Metrics - States " + "="*15)
    states = metrics_dict.get("metrics", {}).get("states", [])
    print("Number of states:", len(states))
    print(json.dumps(states[:showcase_num], indent=4))  # Print only the
    print("...")
    
    # kayeton cam (t ms/#frame/vid time)
    print("="*15 + " Kayeton Cam Data " + "="*15)
    kayeton_cam = metrics_dict.get("kayeton cam (t ms/#frame/vid time)", {})
    print("Number of kayeton cam entries:", len(kayeton_cam))
    # compute average t ms using the first element of each kayeton_cam entry
    t_ms_vals = []
    for entry in kayeton_cam:
        t_ms_vals.append(float(entry[0]))
    intervals = np.diff(t_ms_vals)
    mean_interval = float(np.mean(intervals))
    std_interval = float(np.std(intervals))
    print(f"Average interval between consecutive 't ms': {mean_interval:.3f} ms (±{std_interval:.3f} ms std)")
    print(json.dumps(kayeton_cam[:showcase_num], indent=4))
    print("...")
    
    # wheel
    print("="*15 + " Wheel Data " + "="*15)
    wheel = metrics_dict.get("wheel", {})
    print("Number of wheel entries:", len(wheel))
    print(json.dumps(wheel[:showcase_num], indent=4))
    print("...")

    # clock 100ms
    print("="*15 + " Clock 100ms Data " + "="*15)
    clock_100ms = metrics_dict.get("clock 100ms", {})
    print("Number of clock 100ms entries:", len(clock_100ms))
    print(json.dumps(clock_100ms[:showcase_num], indent=4))
    print("...")
    
    # clock 5min
    print("="*15 + " Clock 5min Data " + "="*15)
    clock_5min = metrics_dict.get("clock 5min", {})
    print("Number of clock 5min entries:", len(clock_5min))
    print(json.dumps(clock_5min[:showcase_num], indent=4))
    print("...")
    
    
_metrics_json_overview(metrics)

In [ ]:
# Generate interactive plot
summary = generate_interactive_session_plot(data_session, output_filename='interactive_session_plot.html', window=10)

## HGF Belief Models

The dataset includes **Hierarchical Gaussian Filter (HGF)** belief model outputs that track how subjects update their beliefs about reward probabilities during the two-arm bandit task. 

The HGF is a computational model for learning in uncertain environments through hierarchical belief updating. It represents beliefs at multiple levels of abstraction:
- **Level 0 (x_0)**: Binary observations (actual outcomes: win/loss)
- **Level 1 (x_1)**: Beliefs about the probability of winning
- **Level 2+ (x_2, x_3, x_4)**: Higher-order beliefs about volatility and uncertainty

Each level tracks:
- `expected_mean`: Prior belief before observing outcome
- `expected_precision`: Confidence in the prior (inverse variance)
- `mean`: Posterior belief after updating with observation
- `observed`: The actual observation at this level
- `precision`: Posterior confidence
- `surprise`: Prediction error (how unexpected the observation was)

The dataset contains three HGF model configurations:
1. **binary2**: 2-level hierarchy (observations + beliefs)
2. **binary3**: 3-level hierarchy (adds meta-beliefs about volatility)
3. **nomasking_1volnode**: 5-level deep hierarchy (full uncertainty modeling)

### Load HGF Data
Load all three HGF model configurations from the `hgf_models` folder.

In [ ]:
# Load HGF model files
hgf_folder = os.path.join(data_session, "hgf_models")

In [ ]:
# Load all three HGF configurations (using PKL for faster loading and precision preservation)
hgf_binary2 = pd.read_pickle(os.path.join(hgf_folder, "20230922_input1_binary2.pkl"))

print("=== HGF Binary2 (2-level hierarchy) ===")
print(f"Shape: {hgf_binary2.shape}")
print(f"Columns: {list(hgf_binary2.columns)}")
print()

print("First few trials of binary2 model:")
print(hgf_binary2.head())

In [ ]:
hgf_binary3 = pd.read_pickle(os.path.join(hgf_folder, "20230922_input1_binary3.pkl"))

print("=== HGF Binary3 (3-level hierarchy) ===")
print(f"Shape: {hgf_binary3.shape}")
print(f"Columns: {list(hgf_binary3.columns)}")
print()

print("First few trials of binary3 model:")
print(hgf_binary3.head())

In [ ]:
hgf_nomasking = pd.read_pickle(os.path.join(hgf_folder, "20230922_input2_nomasking_1volnode.pkl"))

print("=== HGF NoMasking (5-level hierarchy) ===")
print(f"Shape: {hgf_nomasking.shape}")
print(f"Columns: {list(hgf_nomasking.columns)}")
print()

print("First few trials of nomasking model:")
print(hgf_nomasking.head())

**Observation:** All three models track the same number of trials. The key difference is the depth of the hierarchical belief structure:
- Binary2: 2 levels (x_0, x_1)
- Binary3: 3 levels (x_0, x_1, x_2)
- NoMasking: 5 levels (x_0 through x_4)

### HGF Belief Trajectories Over Time
Visualize how beliefs evolve across trials for different hierarchy levels.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def plot_hgf_belief_trajectories(hgf_df, model_name, max_levels=5):
    """
    Plot belief trajectories for all hierarchy levels in an HGF model.
    
    Parameters:
    -----------
    hgf_df : pd.DataFrame
        HGF model output dataframe
    model_name : str
        Name of the model for the title
    max_levels : int
        Maximum number of levels to plot
    """
    # Determine how many levels exist in this model
    levels_present = []
    for i in range(max_levels):
        if f'x_{i}_mean' in hgf_df.columns:
            levels_present.append(i)
    
    n_levels = len(levels_present)
    
    # Create subplots
    fig = make_subplots(
        rows=n_levels, cols=1,
        subplot_titles=[f'Level {i}: {"Observations" if i == 0 else f"Beliefs (x_{i})"}' 
                        for i in levels_present],
        vertical_spacing=0.08
    )
    
    # Plot each level
    for idx, level in enumerate(levels_present):
        row = idx + 1
        
        # Prior (expected) mean
        fig.add_trace(
            go.Scatter(
                x=hgf_df['time'],
                y=hgf_df[f'x_{level}_expected_mean'],
                mode='lines',
                name=f'Level {level} Prior',
                line=dict(dash='dash', width=1.5),
                legendgroup=f'level{level}',
                showlegend=(idx == 0)
            ),
            row=row, col=1
        )
        
        # Posterior mean
        fig.add_trace(
            go.Scatter(
                x=hgf_df['time'],
                y=hgf_df[f'x_{level}_mean'],
                mode='lines',
                name=f'Level {level} Posterior',
                line=dict(width=2),
                legendgroup=f'level{level}',
                showlegend=(idx == 0)
            ),
            row=row, col=1
        )
        
        # Observations (only for level 0)
        if level == 0:
            fig.add_trace(
                go.Scatter(
                    x=hgf_df['time'],
                    y=hgf_df[f'x_{level}_observed'],
                    mode='markers',
                    name=f'Level {level} Observed',
                    marker=dict(size=4, symbol='circle-open'),
                    legendgroup=f'level{level}',
                    showlegend=(idx == 0)
                ),
                row=row, col=1
            )
        
        # Update y-axis labels
        fig.update_yaxes(title_text='Belief State', row=row, col=1)
    
    # Update x-axis label (only on bottom plot)
    fig.update_xaxes(title_text='Time', row=n_levels, col=1)
    
    # Update layout
    fig.update_layout(
        height=250 * n_levels,
        title_text=f'HGF Belief Trajectories: {model_name}',
        hovermode='x unified',
        showlegend=True
    )
    
    return fig

# Plot belief trajectories for all three models
fig_binary2 = plot_hgf_belief_trajectories(hgf_binary2, "Binary2 (2-level)", max_levels=2)
fig_binary2.show()

fig_binary3 = plot_hgf_belief_trajectories(hgf_binary3, "Binary3 (3-level)", max_levels=3)
fig_binary3.show()

fig_nomasking = plot_hgf_belief_trajectories(hgf_nomasking, "NoMasking (5-level)", max_levels=5)
fig_nomasking.show()

**Observations from the plots:**

**Level 0 (Observations):**
- Red vertical lines represent wins (1s), their density shows reward frequency over time
- Prior (dashed blue) and posterior (orange) beliefs track closely, indicating high certainty in the model
- Distinct temporal blocks are visible where win frequency changes, suggesting the task has switching reward contingencies

**Level 1 (Probability Beliefs):**
- **Binary2 & Binary3**: Show nearly identical wave patterns with clear phases:
  - Early learning (0-50): Beliefs start near 0, then rise sharply
  - High reward period (50-100): Beliefs peak around +1.5 to +2, indicating high confidence one arm is better
  - Reversal (100-150): Gradual decline through 0 to negative values as contingencies switch
  - Second high period (175-225): Beliefs rise again to ~+1.5
  - Final decline (225-275): Drop back toward 0 and below
- **NoMasking (5-level)**: Shows more discrete jumps - posterior saturates at 1.0 during high-reward blocks (trials 50-75, 200-225) and drops to near 0 during low periods. This suggests higher certainty/faster learning.

**Higher Levels (Meta-beliefs):**
- **Level 2 (Binary3)**: Shows subtle oscillations (±0.02 range) that track the slower changes in volatility - smoother than Level 1
- **Levels 2-4 (NoMasking)**: 
  - Level 2: Moderate amplitude oscillations mirroring Level 1 block structure
  - Level 3: Large, slow waves indicating meta-volatility tracking (the model learning about how often contingencies change)
  - Level 4: Very slow drift downward (~0 to -0.2), representing the highest-level prior about environmental stability

**Key insight**: The close overlap of prior (dashed) and posterior (solid) lines indicates the model updates beliefs incrementally with high confidence rather than large trial-by-trial jumps.

### Surprise (Prediction Error) Dynamics
Surprise measures how unexpected each observation was. High surprise indicates uncertainty or unexpected events.

In [ ]:
def plot_surprise_metrics(hgf_df, model_name, max_levels=5):
    """
    Plot surprise (prediction error) metrics for each level.
    
    Parameters:
    -----------
    hgf_df : pd.DataFrame
        HGF model output dataframe
    model_name : str
        Name of the model for the title
    max_levels : int
        Maximum number of levels to check
    """
    # Determine which levels have surprise metrics
    surprise_cols = []
    for i in range(max_levels):
        col_name = f'x_{i}_surprise'
        if col_name in hgf_df.columns:
            surprise_cols.append((i, col_name))
    
    # Create figure
    fig = go.Figure()
    
    # Add trace for each level's surprise
    for level, col_name in surprise_cols:
        fig.add_trace(
            go.Scatter(
                x=hgf_df['time'],
                y=hgf_df[col_name],
                mode='lines',
                name=f'Level {level} Surprise',
                line=dict(width=2)
            )
        )
    
    # Add total surprise if available
    if 'total_surprise' in hgf_df.columns:
        fig.add_trace(
            go.Scatter(
                x=hgf_df['time'],
                y=hgf_df['total_surprise'],
                mode='lines',
                name='Total Surprise',
                line=dict(width=3, dash='dot', color='black')
            )
        )
    
    # Update layout
    fig.update_layout(
        title=f'Surprise (Prediction Error) Dynamics: {model_name}',
        xaxis_title='Time',
        yaxis_title='Surprise',
        hovermode='x unified',
        height=500
    )
    
    return fig

# Plot surprise metrics for all three models
fig_surprise_binary2 = plot_surprise_metrics(hgf_binary2, "Binary2 (2-level)", max_levels=2)
fig_surprise_binary2.show()

fig_surprise_binary3 = plot_surprise_metrics(hgf_binary3, "Binary3 (3-level)", max_levels=3)
fig_surprise_binary3.show()

fig_surprise_nomasking = plot_surprise_metrics(hgf_nomasking, "NoMasking (5-level)", max_levels=5)
fig_surprise_nomasking.show()

**Observations from the plots:**

**Level 0 Surprise (Observation-level prediction errors):**
- Identical spiky pattern across all three models, oscillating between 0 and ~1.5
- Rapid trial-by-trial fluctuations reflecting binary prediction errors when outcomes don't match expectations
- Regular drop-downs to near 0 when predictions are correct, sharp spikes to ~1-1.5 when incorrect

**Level 1 Surprise (Belief-level prediction errors):**
- **Binary2 & Binary3**: Very smooth and stable, staying consistently around 0.3-0.5 throughout
  - Indicates steady, gradual belief updating without major shocks
- **NoMasking (5-level)**: Dramatically different with multiple large spikes:
  - Spike to ~2.7 around trial 40
  - Spike to ~2.7 around trial 115  
  - **Massive spike to ~4-5 around trial 165** (unprecedented surprise in beliefs)
  - Spike to ~2 around trial 215
  - These spikes indicate moments when the model detects unexpected shifts in reward contingencies

**Higher-Level Surprise:**
- **Level 2 (Binary3)**: Completely flat horizontal line at ~1.0-1.2
  - Constant surprise contribution, not adapting to environmental changes
- **NoMasking Levels 2-4**: All relatively flat and low (0-1 range)
  - Higher levels remain stable, providing baseline priors while lower levels react

**Total Surprise:**
- **Binary2**: Ranges 0.5-2.7, dominated by Level 0 spikes with moderate baseline
- **Binary3**: Ranges 1.5-3.8, higher baseline due to Level 2's constant ~1.0 contribution
- **NoMasking**: Ranges 2.5-9 with extreme variability:
  - **Massive spike to ~9 at trial 165** when multiple levels detect environmental change simultaneously
  - Baseline is higher (~2.5-3) due to summing across 5 levels
  - Much more sensitive to detecting contingency reversals

**Key insights:**
- Total surprise increases with hierarchical depth (2-level < 3-level < 5-level) due to summing across more levels
- The 5-level model's Level 1 surprise spikes pinpoint exact moments of contingency reversals (trials ~40, 115, 165, 215)
- Simpler models (Binary2/3) maintain stable Level 1 beliefs, while the 5-level model shows adaptive sensitivity to environmental changes

### Precision (Confidence) Evolution
Precision represents the inverse of variance - higher precision means more confident beliefs.

In [ ]:
def plot_precision_evolution(hgf_df, model_name, max_levels=5):
    """
    Plot precision (confidence) evolution for each level.
    
    Parameters:
    -----------
    hgf_df : pd.DataFrame
        HGF model output dataframe
    model_name : str
        Name of the model for the title
    max_levels : int
        Maximum number of levels to check
    """
    # Determine which levels have precision metrics
    levels_present = []
    for i in range(max_levels):
        if f'x_{i}_precision' in hgf_df.columns:
            levels_present.append(i)
    
    n_levels = len(levels_present)
    
    # Create subplots
    fig = make_subplots(
        rows=n_levels, cols=1,
        subplot_titles=[f'Level {i}: Precision Evolution' for i in levels_present],
        vertical_spacing=0.08
    )
    
    # Plot each level
    for idx, level in enumerate(levels_present):
        row = idx + 1
        
        # Expected precision
        fig.add_trace(
            go.Scatter(
                x=hgf_df['time'],
                y=hgf_df[f'x_{level}_expected_precision'],
                mode='lines',
                name=f'Level {level} Expected',
                line=dict(dash='dash', width=1.5),
                legendgroup=f'level{level}',
                showlegend=(idx == 0)
            ),
            row=row, col=1
        )
        
        # Posterior precision
        fig.add_trace(
            go.Scatter(
                x=hgf_df['time'],
                y=hgf_df[f'x_{level}_precision'],
                mode='lines',
                name=f'Level {level} Posterior',
                line=dict(width=2),
                legendgroup=f'level{level}',
                showlegend=(idx == 0)
            ),
            row=row, col=1
        )
        
        # Update y-axis labels
        fig.update_yaxes(title_text='Precision', row=row, col=1)
    
    # Update x-axis label (only on bottom plot)
    fig.update_xaxes(title_text='Time', row=n_levels, col=1)
    
    # Update layout
    fig.update_layout(
        height=250 * n_levels,
        title_text=f'Precision (Confidence) Evolution: {model_name}',
        hovermode='x unified',
        showlegend=True
    )
    
    return fig

# Plot precision for binary3 model as an example
fig_precision = plot_precision_evolution(hgf_binary3, "Binary3 (3-level)", max_levels=3)
fig_precision.show()

**Observations from the plot (Binary3 3-level model):**

**Level 0 Precision (Observation-level confidence):**
- Highly variable, oscillating rapidly between 0.1 and 0.25 throughout the entire session
- No clear trend over time - maintains constant variability
- Expected (dashed) and posterior (solid) lines nearly overlap, indicating minimal updating
- Reflects inherent uncertainty in binary observations - confidence doesn't accumulate at this level

**Level 1 Precision (Belief-level confidence):**
- **Rapid initial increase**: Jumps from ~1.0 to ~2.5-3.0 within the first 20 trials as the model accumulates evidence
- **Stabilizes with structured oscillations**: After initial learning, precision oscillates between 2.5-3.2
- Oscillations correlate with belief changes - precision dips during contingency reversals (around trials 50, 100, 150, 200) when uncertainty increases
- Expected vs posterior precision track very closely with minimal separation
- Shows classic learning curve: rapid confidence gain early, then modulation based on environmental stability

**Level 2 Precision (Meta-level volatility confidence):**
- **Monotonic decrease** from 1.0 to ~0.6 over the entire session
- Very smooth trajectory with no oscillations
- Decreasing precision indicates the model is becoming LESS confident about volatility estimates
- This suggests the model is learning that the environment is more volatile than initially assumed
- Represents meta-learning about the changeability of reward contingencies

**Key insights:**
- Precision dynamics differ dramatically across hierarchical levels: Level 0 = constant noise, Level 1 = learning then tracking, Level 2 = meta-learning about volatility
- The gap between expected and posterior precision is minimal at all levels, indicating the model's updates are well-calibrated
- Level 2's decreasing precision explains why Level 1 can maintain adaptive oscillations - the model learns to expect environmental changes

### Model Comparison: Hierarchical Depth Effects
Compare belief trajectories across different hierarchical depths (2 vs 3 vs 5 levels).

In [ ]:
# Compare Level 1 beliefs across all three models
fig_comparison = go.Figure()

fig_comparison.add_trace(
    go.Scatter(
        x=hgf_binary2['time'],
        y=hgf_binary2['x_1_mean'],
        mode='lines',
        name='Binary2 (2-level)',
        line=dict(width=2)
    )
)

fig_comparison.add_trace(
    go.Scatter(
        x=hgf_binary3['time'],
        y=hgf_binary3['x_1_mean'],
        mode='lines',
        name='Binary3 (3-level)',
        line=dict(width=2)
    )
)

fig_comparison.add_trace(
    go.Scatter(
        x=hgf_nomasking['time'],
        y=hgf_nomasking['x_1_mean'],
        mode='lines',
        name='NoMasking (5-level)',
        line=dict(width=2)
    )
)

# Add observations for reference
fig_comparison.add_trace(
    go.Scatter(
        x=hgf_binary2['time'],
        y=hgf_binary2['x_0_observed'],
        mode='markers',
        name='Observations',
        marker=dict(size=4, symbol='circle-open', color='black'),
        opacity=0.4
    )
)

fig_comparison.update_layout(
    title='Comparison of Level 1 Beliefs Across HGF Models',
    xaxis_title='Time',
    yaxis_title='Belief (Probability)',
    hovermode='x unified',
    height=500,
    legend=dict(x=1.05, y=1)
)

fig_comparison.show()

# Compare total surprise across models
fig_surprise_comp = go.Figure()

fig_surprise_comp.add_trace(
    go.Scatter(
        x=hgf_binary2['time'],
        y=hgf_binary2['total_surprise'],
        mode='lines',
        name='Binary2 (2-level)',
        line=dict(width=2)
    )
)

fig_surprise_comp.add_trace(
    go.Scatter(
        x=hgf_binary3['time'],
        y=hgf_binary3['total_surprise'],
        mode='lines',
        name='Binary3 (3-level)',
        line=dict(width=2)
    )
)

fig_surprise_comp.add_trace(
    go.Scatter(
        x=hgf_nomasking['time'],
        y=hgf_nomasking['total_surprise'],
        mode='lines',
        name='NoMasking (5-level)',
        line=dict(width=2)
    )
)

fig_surprise_comp.update_layout(
    title='Comparison of Total Surprise Across HGF Models',
    xaxis_title='Time',
    yaxis_title='Total Surprise',
    hovermode='x unified',
    height=500,
    legend=dict(x=1.05, y=1)
)

fig_surprise_comp.show()

**Observations from the comparison plots:**

**Level 1 Beliefs (Probability Estimates):**
- **Binary2 and Binary3**: Nearly identical trajectories, appearing as overlapping smooth waves
  - Both oscillate continuously between -2 and +2
  - Gradual transitions through belief states showing incremental updating
  - Wave peaks around +1.5 to +2.0 during high-reward blocks, troughs around -1.5 during low-reward blocks
- **NoMasking (5-level)**: Dramatically different behavior - discrete, step-like pattern
  - Saturates at 1.0 during high-reward blocks (trials 0-25, 50-100, 200-225)
  - Drops to near 0 during low-reward periods (trials 25-50, 100-175, 225-275)
  - More **reactive**, not smoother - makes rapid, binary-like jumps between states
  - Higher certainty: commits fully to beliefs (0 or 1) rather than intermediate probability estimates

**Total Surprise (Model Uncertainty):**
- **Hierarchy depth increases surprise, contrary to initial expectation:**
  - **Binary2 (blue)**: Lowest surprise, ranges 0.5-2.7
  - **Binary3 (red)**: Moderate surprise, ranges 1.5-3.8
  - **NoMasking (green)**: Highest surprise, ranges 2.5-9 with massive spike to ~9 around trial 165
- Deeper hierarchies sum surprise across more levels, resulting in higher total values
- NoMasking's extreme spikes indicate the model detects contingency reversals more sensitively

**Model Choice Implications:**
- **2-level (Binary2)**: Lowest computational complexity, moderate reactivity, minimal surprise accumulation
- **3-level (Binary3)**: Adds volatility layer, slightly higher baseline surprise due to additional level
- **5-level (NoMasking)**: 
  - Most sensitive to environmental changes (highest surprise spikes)
  - Makes more confident, binary-like probability estimates (0 or 1)
  - Highest computational cost and surprise accumulation
  - Better at **detecting** changes, not necessarily smoother predictions

**Key insight**: Deeper hierarchies don't necessarily mean "better fit" in terms of lower surprise. Instead, they provide richer uncertainty representation and sharper change detection at the cost of higher total surprise.

### Summary Statistics
Compute key statistics for HGF belief model outputs.

In [ ]:
def hgf_summary_statistics(hgf_df, model_name):
    """Compute summary statistics for HGF model outputs."""
    
    print(f"{'='*60}")
    print(f"HGF Summary Statistics: {model_name}")
    print(f"{'='*60}")
    
    # Number of trials
    n_trials = len(hgf_df)
    print(f"Number of trials: {n_trials}")
    
    # Observation statistics
    if 'x_0_observed' in hgf_df.columns:
        n_wins = hgf_df['x_0_observed'].sum()
        win_rate = n_wins / n_trials
        print(f"Win rate: {win_rate:.2%} ({int(n_wins)} wins out of {n_trials} trials)")
    
    # Level 1 belief statistics
    if 'x_1_mean' in hgf_df.columns:
        belief_mean = hgf_df['x_1_mean'].mean()
        belief_std = hgf_df['x_1_mean'].std()
        belief_range = (hgf_df['x_1_mean'].min(), hgf_df['x_1_mean'].max())
        print(f"\nLevel 1 Belief (probability estimate):")
        print(f"  Mean: {belief_mean:.3f}")
        print(f"  Std:  {belief_std:.3f}")
        print(f"  Range: [{belief_range[0]:.3f}, {belief_range[1]:.3f}]")
    
    # Surprise statistics
    if 'total_surprise' in hgf_df.columns:
        surprise_mean = hgf_df['total_surprise'].mean()
        surprise_std = hgf_df['total_surprise'].std()
        surprise_max = hgf_df['total_surprise'].max()
        
        # Early vs late surprise (first half vs second half)
        mid_point = n_trials // 2
        early_surprise = hgf_df['total_surprise'].iloc[:mid_point].mean()
        late_surprise = hgf_df['total_surprise'].iloc[mid_point:].mean()
        
        print(f"\nTotal Surprise:")
        print(f"  Mean: {surprise_mean:.3f}")
        print(f"  Std:  {surprise_std:.3f}")
        print(f"  Max:  {surprise_max:.3f}")
        print(f"  Early trials (first half): {early_surprise:.3f}")
        print(f"  Late trials (second half):  {late_surprise:.3f}")
        print(f"  Learning effect: {(early_surprise - late_surprise):.3f} reduction")
    
    # Precision statistics (confidence)
    if 'x_1_precision' in hgf_df.columns:
        precision_mean = hgf_df['x_1_precision'].mean()
        precision_final = hgf_df['x_1_precision'].iloc[-1]
        precision_initial = hgf_df['x_1_precision'].iloc[0]
        
        print(f"\nLevel 1 Precision (confidence):")
        print(f"  Mean: {precision_mean:.3f}")
        print(f"  Initial: {precision_initial:.3f}")
        print(f"  Final: {precision_final:.3f}")
        print(f"  Change: +{(precision_final - precision_initial):.3f}")
    
    print()

# Print summary statistics for all three models
hgf_summary_statistics(hgf_binary2, "Binary2 (2-level)")
hgf_summary_statistics(hgf_binary3, "Binary3 (3-level)")
hgf_summary_statistics(hgf_nomasking, "NoMasking (5-level)")